In [34]:
import pandas as pd
from transformers import AutoTokenizer


data = pd.read_parquet('/Users/yavuzlule/Desktop/bsc-relish/data/external/recipe1m/full_dataset_labeled.parquet')
tokenizer = AutoTokenizer.from_pretrained("/Users/yavuzlule/Desktop/bsc-relish/results/xlmroberta-base/2026-05-25_11-01-46")

data.head()

/Users/yavuzlule/Desktop/bsc-relish/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,chunk_text,label
0,"In a heavy 2-quart saucepan, mix brown sugar, ...",1.0
1,"Place chipped beef on bottom of baking dish."",...",1.0
2,"In a slow cooker, combine all ingredients. Cov...",1.0
3,"Boil and debone chicken."", ""Put bite size piec...",1.0
4,Combine first four ingredients and press in 13...,1.0


In [3]:
rows = []
MAX_WORDS = 256

for row_idx, row in data.iterrows():
    words = str(row["chunk_text"]).split()

    # Split into chunks of max 256 words
    chunks = [
        words[i:i + MAX_WORDS]
        for i in range(0, len(words), MAX_WORDS)
    ]

    for chunk_idx, chunk in enumerate(chunks):
        rows.append({
            "original_row": row_idx,
            "chunk_index": chunk_idx,
            "chunk_text": " ".join(chunk)
        })

chunked_df = pd.DataFrame(rows)

print(chunked_df.head())

   original_row  chunk_index  \
0             0            0   
1             1            0   
2             2            0   
3             3            0   
4             4            0   

                                          chunk_text  
0  In a heavy 2-quart saucepan, mix brown sugar, ...  
1  Place chipped beef on bottom of baking dish.",...  
2  In a slow cooker, combine all ingredients. Cov...  
3  Boil and debone chicken.", "Put bite size piec...  
4  Combine first four ingredients and press in 13...  


In [27]:
import re

def clean_recipe_text(text: str) -> str:
    # Remove surrounding quotes and stray quote characters
    text = text.replace('"', '').replace("'", "")


    # Replace list-style separators like ", " between instructions with sentence breaks
    # Heuristic: turn sequences like ", Turn", ", Cover" into ". Turn", ". Cover"
    text = re.sub(r',\s*(?=[A-Z])', '. ', text)

    # Remove double periods or broken punctuation
    text = re.sub(r'\.+', '.', text)

    # Clean spacing
    text = re.sub(r'\s+', ' ', text).strip()

    # Ensure proper spacing after periods
    text = re.sub(r'\.\s*', '. ', text).strip()

    return text

In [28]:
import os

def export_column_cells_to_txt(df, column, start_index, end_index, output_folder):
    """
    Writes each cell in df[column] from start_index to end_index into separate .txt files.
    
    Args:
        df (pd.DataFrame): input dataframe
        column (str): column name to export
        start_index (int): inclusive start index (iloc-based)
        end_index (int): inclusive end index (iloc-based)
        output_folder (str): directory to save txt files
    """
    
    # Ensure folder exists
    os.makedirs(output_folder, exist_ok=True)
    
    # Slice dataframe (inclusive end_index like user requested)
    subset = df.iloc[start_index:end_index + 1]
    
    for i, value in enumerate(subset[column], start=start_index):
        file_path = os.path.join(output_folder, f"{i}.txt")
        
        # Handle NaN / non-string safely
        text = "" if value is None else str(clean_recipe_text(value))
        
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(text)

In [29]:
text = 'Grate potatoes, draining excess water.\tGrate onion (and carrot), add to potatoes.", "Mix in remaining ingredients and pour into an 8 inch or 9 inch square greased pan.", "Sprinkle with paprika to aid in browning.", "Bake at 375 for 1 hour or until golden brown on top.", "Serves 2 to 3 - Can be doubled for a 9 x 13 pan.'

In [33]:
languages = ["es", "fr", "de", "it", "nl", "en"]

for i in range(len(languages)):
    export_column_cells_to_txt(
        df=chunked_df,
        column="chunk_text",
        start_index=i*2000,
        end_index=(i+1)*2000-1,  # end_index is inclusive, so subtract 1
        output_folder=f"chunked_recipes_2k_en/{languages[i]}"
    )